# Word Segmentation — BiLSTM Ensemble + Final Retrain

Upgraded from baseline with:
1. **4-seed ensemble** — train multiple seeds, average logits before decoding
2. **2-phase training** — Phase 1 finds best epoch on eval; Phase 2 retrains on train+eval
3. **Larger hidden_dim (320)** — more model capacity
4. **Vocab from train+eval** — no unseen chars at final train time
5. **Constrained Viterbi decode** on averaged logits (no CRF needed for ensemble)

In [1]:
# =============================================================
# 1. Imports & Config
# =============================================================
import os, gc, random, unicodedata
from pathlib import Path
from collections import Counter
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
print("DEVICE:", DEVICE)

CFG = {
    "seeds":             [42, 52, 62],   # multi-seed ensemble
    "batch_size":        64,
    "valid_batch_size":  128,
    "phase1_epochs":     6,
    "final_extra_epochs":1,
    "lr":                2e-3,
    "weight_decay":      1e-4,
    "char_emb_dim":      128,
    "type_emb_dim":      16,
    "hidden_dim":        320,
    "num_layers":        2,
    "dropout":           0.20,
    "max_grad_norm":     1.0,
    "label_smoothing":   0.03,
    "min_char_freq":     1,
    "num_workers":       2,
}
CFG


DEVICE: cuda


{'seeds': [42, 52, 62],
 'batch_size': 64,
 'valid_batch_size': 128,
 'phase1_epochs': 6,
 'final_extra_epochs': 1,
 'lr': 0.002,
 'weight_decay': 0.0001,
 'char_emb_dim': 128,
 'type_emb_dim': 16,
 'hidden_dim': 320,
 'num_layers': 2,
 'dropout': 0.2,
 'max_grad_norm': 1.0,
 'label_smoothing': 0.03,
 'min_char_freq': 1,
 'num_workers': 2}

In [2]:
# =============================================================
# 2. Paths — auto-locate on Kaggle
# =============================================================
INPUT_DIR = Path("/kaggle/input")

def find_dir(dirname: str) -> Path:
    matches = [p for p in INPUT_DIR.rglob(dirname) if p.is_dir()]
    if not matches:
        raise FileNotFoundError(f"Cannot find folder: {dirname}")
    return sorted(matches, key=lambda p: len(str(p)))[0]

def find_file(filename: str) -> Path:
    matches = [p for p in INPUT_DIR.rglob(filename) if p.is_file()]
    if not matches:
        raise FileNotFoundError(f"Cannot find file: {filename}")
    return sorted(matches, key=lambda p: len(str(p)))[0]

corpus_dir      = find_dir("LST20_Corpus")
ws_test_path    = find_file("ws_test.txt")
sample_sub_path = find_file("ws_sample_submission.csv")
ws_list_path    = find_file("ws_list.txt")

print("corpus_dir      :", corpus_dir)
print("ws_test_path    :", ws_test_path)
print("sample_sub_path :", sample_sub_path)


corpus_dir      : /kaggle/input/datasets/rainyze/lst20corpusdatasetwindowusable/LST20_Corpus
ws_test_path    : /kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation/ws_test.txt
sample_sub_path : /kaggle/input/competitions/super-ai-engineer-ss-6-word-segmentation/ws_sample_submission.csv


In [3]:
# =============================================================
# 3. Labels & Utilities
# =============================================================
LABEL2ID = {"B_WORD": 0, "I_WORD": 1, "E_WORD": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
PAD_ID    = 0
UNK_ID    = 1

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def token_to_tags(token: str) -> List[str]:
    if len(token) == 1:
        return ["B_WORD"]
    return ["B_WORD"] + ["I_WORD"] * (len(token) - 2) + ["E_WORD"]

def char_type_id(ch: str) -> int:
    """6-class char type: 0=PAD 1=digit 2=latin 3=punct/symbol 4=thai 5=other"""
    if ch.isdigit():
        return 1
    if ("A" <= ch <= "Z") or ("a" <= ch <= "z"):
        return 2
    cat = unicodedata.category(ch)
    if cat.startswith("P") or cat.startswith("S"):
        return 3
    if "\u0E00" <= ch <= "\u0E7F":
        return 4
    return 5


In [4]:
# =============================================================
# 4. Load LST20 corpus
# =============================================================
def read_lst20_sequences(folder: Path) -> List[Tuple[str, List[str]]]:
    sequences = []
    chars, labels = [], []
    for path in sorted(folder.glob("*.txt")):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.rstrip("\n")
                if line == "":
                    if chars:
                        sequences.append(("".join(chars), labels.copy()))
                        chars.clear(); labels.clear()
                    continue
                token = line.split("\t")[0]
                if token == "_":      # whitespace token in LST20
                    continue
                chars.extend(list(token))
                labels.extend(token_to_tags(token))
        if chars:
            sequences.append(("".join(chars), labels.copy()))
            chars.clear(); labels.clear()
    return sequences

train_sequences = read_lst20_sequences(corpus_dir / "train")
eval_sequences  = read_lst20_sequences(corpus_dir / "eval")
full_sequences  = train_sequences + eval_sequences   # used in phase 2 & vocab

print("train sequences:", len(train_sequences))
print("eval  sequences:", len(eval_sequences))
print("full  sequences:", len(full_sequences))


train sequences: 63310
eval  sequences: 5620
full  sequences: 68930


In [5]:
# =============================================================
# 5. Build vocab from FULL trainable data (train + eval)
# =============================================================
def build_vocab(sequences):
    char_counter = Counter()
    for text, _ in sequences:
        char_counter.update(text)
    stoi = {"<PAD>": PAD_ID, "<UNK>": UNK_ID}
    for ch, freq in char_counter.items():
        if freq >= CFG["min_char_freq"]:
            stoi[ch] = len(stoi)
    itos = {i: s for s, i in stoi.items()}
    return stoi, itos, char_counter

stoi, itos, char_counter = build_vocab(full_sequences)
print("vocab size:", len(stoi))
print("most common:", char_counter.most_common(10))


vocab size: 181
most common: [('า', 787582), ('น', 627898), ('ร', 598227), ('่', 471996), ('ก', 456553), ('อ', 421605), ('ง', 391104), ('เ', 377358), ('้', 370098), ('ม', 362705)]


In [6]:
# =============================================================
# 6. Dataset & DataLoader
# =============================================================
class WSDataset(Dataset):
    def __init__(self, sequences, stoi):
        self.sequences = sequences
        self.stoi = stoi

    def __len__(self): return len(self.sequences)

    def __getitem__(self, idx):
        text, labels = self.sequences[idx]
        char_ids  = [self.stoi.get(ch, UNK_ID) for ch in text]
        type_ids  = [char_type_id(ch) for ch in text]
        label_ids = [LABEL2ID[tag] for tag in labels]
        return (
            torch.tensor(char_ids,  dtype=torch.long),
            torch.tensor(type_ids,  dtype=torch.long),
            torch.tensor(label_ids, dtype=torch.long),
            text,
        )

def collate_fn(batch):
    char_ids, type_ids, label_ids, texts = zip(*batch)
    lengths = torch.tensor([len(x) for x in char_ids], dtype=torch.long)
    max_len = int(lengths.max())
    x = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
    t = torch.zeros((len(batch), max_len), dtype=torch.long)
    y = torch.full((len(batch), max_len), -100, dtype=torch.long)
    mask = torch.zeros((len(batch), max_len), dtype=torch.bool)
    for i, (xi, ti, yi) in enumerate(zip(char_ids, type_ids, label_ids)):
        n = len(xi)
        x[i, :n] = xi; t[i, :n] = ti; y[i, :n] = yi; mask[i, :n] = True
    return x, t, y, lengths, mask, texts

# ---------- Test dataset ----------
class WSTestDataset(Dataset):
    def __init__(self, chunks, stoi):
        self.chunks = chunks
        self.stoi = stoi

    def __len__(self): return len(self.chunks)

    def __getitem__(self, idx):
        text, original_ids = self.chunks[idx]
        x = [self.stoi.get(ch, UNK_ID) for ch in text]
        t = [char_type_id(ch) for ch in text]
        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(t, dtype=torch.long),
            text,
            original_ids,
            idx,
        )

def test_collate_fn(batch):
    char_ids, type_ids, texts, original_ids, chunk_indices = zip(*batch)
    lengths = torch.tensor([len(x) for x in char_ids], dtype=torch.long)
    max_len = int(lengths.max())
    x    = torch.full((len(batch), max_len), PAD_ID, dtype=torch.long)
    t    = torch.zeros((len(batch), max_len), dtype=torch.long)
    mask = torch.zeros((len(batch), max_len), dtype=torch.bool)
    for i, (xi, ti) in enumerate(zip(char_ids, type_ids)):
        n = len(xi); x[i, :n] = xi; t[i, :n] = ti; mask[i, :n] = True
    return x, t, lengths, mask, texts, original_ids, chunk_indices


In [7]:
# =============================================================
# 7. BiLSTM Model  (hidden_dim=320, no CRF — ensemble on logits)
# =============================================================
class BiLSTMTagger(nn.Module):
    def __init__(self, vocab_size, num_char_types=6, num_labels=3,
                 char_emb_dim=128, type_emb_dim=16,
                 hidden_dim=320, num_layers=2, dropout=0.2):
        super().__init__()
        self.char_emb = nn.Embedding(vocab_size, char_emb_dim, padding_idx=PAD_ID)
        self.type_emb = nn.Embedding(num_char_types, type_emb_dim, padding_idx=0)
        lstm_dropout  = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=char_emb_dim + type_emb_dim,
            hidden_size=hidden_dim // 2,
            num_layers=num_layers,
            dropout=lstm_dropout,
            bidirectional=True,
            batch_first=True,
        )
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim, num_labels)

    def forward(self, x, t, lengths):
        emb = torch.cat([self.char_emb(x), self.type_emb(t)], dim=-1)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        return self.classifier(self.dropout(out))   # (B, T, 3)

def build_model():
    return BiLSTMTagger(
        vocab_size=len(stoi),
        char_emb_dim=CFG["char_emb_dim"],
        type_emb_dim=CFG["type_emb_dim"],
        hidden_dim=CFG["hidden_dim"],
        num_layers=CFG["num_layers"],
        dropout=CFG["dropout"],
    ).to(DEVICE)

def make_autocast():
    return torch.amp.autocast("cuda", enabled=USE_AMP)

def make_scaler():
    return torch.amp.GradScaler("cuda", enabled=USE_AMP)

print(f"Model params: {sum(p.numel() for p in build_model().parameters()):,}")


Model params: 1,032,867


In [8]:
# =============================================================
# 8. Constrained Viterbi decoder  (B→B/I/E, I→I/E, E→B)
# =============================================================
ALLOWED_NEXT = {
    -1: [0],          # START → B
     0: [0, 1, 2],    # B     → B / I / E
     1: [1, 2],       # I     → I / E
     2: [0],          # E     → B
}
END_ALLOWED = {0, 2}

def constrained_decode_single(logits_np: np.ndarray) -> List[int]:
    L   = logits_np.shape[0]
    NEG = -1e18
    dp   = np.full((L, 3), NEG, dtype=np.float64)
    back = np.full((L, 3), -1,  dtype=np.int64)

    for s in ALLOWED_NEXT[-1]:
        dp[0, s] = logits_np[0, s]

    for i in range(1, L):
        for s in range(3):
            best_score, best_prev = NEG, -1
            for p in range(3):
                if s not in ALLOWED_NEXT[p]: continue
                score = dp[i-1, p] + logits_np[i, s]
                if score > best_score:
                    best_score, best_prev = score, p
            dp[i, s]   = best_score
            back[i, s] = best_prev

    last = max(END_ALLOWED, key=lambda s: dp[L-1, s])
    seq  = [last]
    for i in range(L-1, 0, -1):
        last = back[i, last]; seq.append(last)
    return list(reversed(seq))


In [9]:
# =============================================================
# 9. Train / eval helpers
# =============================================================
def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    for x, t, y, lengths, mask, texts in loader:
        x = x.to(DEVICE, non_blocking=True)
        t = t.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with make_autocast():
            logits = model(x, t, lengths)
            loss   = F.cross_entropy(
                logits.reshape(-1, 3), y.reshape(-1),
                ignore_index=-100,
                label_smoothing=CFG["label_smoothing"],
            )
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["max_grad_norm"])
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_true, all_pred = [], []
    for x, t, y, lengths, mask, texts in loader:
        x = x.to(DEVICE, non_blocking=True)
        t = t.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        with make_autocast():
            logits = model(x, t, lengths)
            loss   = F.cross_entropy(logits.reshape(-1, 3), y.reshape(-1), ignore_index=-100)
        total_loss += loss.item()
        logits_cpu = logits.detach().float().cpu().numpy()
        y_cpu      = y.detach().cpu().numpy()
        for i, seq_len in enumerate(lengths.cpu().numpy()):
            pred_ids = constrained_decode_single(logits_cpu[i, :seq_len])
            true_ids = y_cpu[i, :seq_len].tolist()
            all_pred.extend(pred_ids); all_true.extend(true_ids)
    macro_f1 = f1_score(all_true, all_pred, average="macro")
    return total_loss / max(len(loader), 1), macro_f1


In [10]:
# =============================================================
# 10. PHASE 1 — Find best epoch per seed (train only)
# =============================================================
phase1_records = []
phase1_best    = []

train_ds = WSDataset(train_sequences, stoi)
eval_ds  = WSDataset(eval_sequences,  stoi)

for seed in CFG["seeds"]:
    print("=" * 70)
    print(f"PHASE 1 | seed = {seed}")
    set_seed(seed)

    train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                              collate_fn=collate_fn, num_workers=CFG["num_workers"],
                              pin_memory=torch.cuda.is_available())
    eval_loader  = DataLoader(eval_ds, batch_size=CFG["valid_batch_size"], shuffle=False,
                              collate_fn=collate_fn, num_workers=CFG["num_workers"],
                              pin_memory=torch.cuda.is_available())

    model     = build_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=1)
    scaler    = make_scaler()

    best_f1, best_epoch = -1.0, 1
    best_ckpt_path = f"/kaggle/working/phase1_seed_{seed}.pt"

    for epoch in range(1, CFG["phase1_epochs"] + 1):
        train_loss          = train_one_epoch(model, train_loader, optimizer, scaler)
        eval_loss, eval_f1  = evaluate(model, eval_loader)
        scheduler.step(eval_f1)

        phase1_records.append({"seed": seed, "epoch": epoch,
                                "train_loss": train_loss, "eval_loss": eval_loss,
                                "eval_macro_f1": eval_f1,
                                "lr": optimizer.param_groups[0]["lr"]})
        print(f"seed={seed} | epoch={epoch:02d} | train_loss={train_loss:.5f} | "
              f"eval_loss={eval_loss:.5f} | eval_macro_f1={eval_f1:.5f} | "
              f"lr={optimizer.param_groups[0]['lr']:.2e}")

        if eval_f1 > best_f1:
            best_f1, best_epoch = eval_f1, epoch
            torch.save({"model_state_dict": model.state_dict(),
                        "seed": seed, "best_epoch": best_epoch, "best_f1": best_f1},
                       best_ckpt_path)
            print(f"  -> saved best checkpoint (epoch {best_epoch}, f1={best_f1:.5f})")

    phase1_best.append({"seed": seed, "best_epoch": best_epoch,
                         "best_eval_macro_f1": best_f1, "phase1_ckpt": best_ckpt_path})

    del model, optimizer, scheduler, scaler, train_loader, eval_loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

phase1_best_df = pd.DataFrame(phase1_best)
print("\nBest per seed:")
display(phase1_best_df)


PHASE 1 | seed = 42
seed=42 | epoch=01 | train_loss=0.20507 | eval_loss=0.07110 | eval_macro_f1=0.97968 | lr=2.00e-03
  -> saved best checkpoint (epoch 1, f1=0.97968)
seed=42 | epoch=02 | train_loss=0.14909 | eval_loss=0.06228 | eval_macro_f1=0.98320 | lr=2.00e-03
  -> saved best checkpoint (epoch 2, f1=0.98320)
seed=42 | epoch=03 | train_loss=0.14213 | eval_loss=0.05683 | eval_macro_f1=0.98478 | lr=2.00e-03
  -> saved best checkpoint (epoch 3, f1=0.98478)
seed=42 | epoch=04 | train_loss=0.13864 | eval_loss=0.05593 | eval_macro_f1=0.98551 | lr=2.00e-03
  -> saved best checkpoint (epoch 4, f1=0.98551)
seed=42 | epoch=05 | train_loss=0.13644 | eval_loss=0.05611 | eval_macro_f1=0.98563 | lr=2.00e-03
  -> saved best checkpoint (epoch 5, f1=0.98563)
seed=42 | epoch=06 | train_loss=0.13492 | eval_loss=0.05416 | eval_macro_f1=0.98588 | lr=2.00e-03
  -> saved best checkpoint (epoch 6, f1=0.98588)
PHASE 1 | seed = 52
seed=52 | epoch=01 | train_loss=0.20592 | eval_loss=0.07163 | eval_macro_f1=0.

,seed,best_epoch,best_eval_macro_f1,phase1_ckpt
0,42,6,0.985880,/kaggle/working/phase1_seed_42.pt
1,52,5,0.986325,/kaggle/working/phase1_seed_52.pt
2,62,6,0.985993,/kaggle/working/phase1_seed_62.pt


In [11]:
# =============================================================
# 11. PHASE 2 — Retrain each seed on train + eval
# =============================================================
full_ds     = WSDataset(full_sequences, stoi)
final_ckpts = []

for row in phase1_best:
    seed         = row["seed"]
    final_epochs = int(row["best_epoch"]) + int(CFG["final_extra_epochs"])

    print("=" * 70)
    print(f"PHASE 2 | seed = {seed} | final_epochs = {final_epochs}")
    set_seed(seed)

    full_loader = DataLoader(full_ds, batch_size=CFG["batch_size"], shuffle=True,
                             collate_fn=collate_fn, num_workers=CFG["num_workers"],
                             pin_memory=torch.cuda.is_available())

    model     = build_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    scaler    = make_scaler()

    for epoch in range(1, final_epochs + 1):
        train_loss = train_one_epoch(model, full_loader, optimizer, scaler)
        print(f"seed={seed} | final_epoch={epoch:02d}/{final_epochs:02d} | train_loss={train_loss:.5f}")

    final_ckpt_path = f"/kaggle/working/final_seed_{seed}.pt"
    torch.save({"model_state_dict": model.state_dict(),
                "seed": seed, "final_epochs": final_epochs}, final_ckpt_path)
    final_ckpts.append(final_ckpt_path)
    print(f"  -> saved {final_ckpt_path}")

    del model, optimizer, scaler, full_loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print("\nFinal checkpoints:", final_ckpts)


PHASE 2 | seed = 42 | final_epochs = 7
seed=42 | final_epoch=01/07 | train_loss=0.20212
seed=42 | final_epoch=02/07 | train_loss=0.14869
seed=42 | final_epoch=03/07 | train_loss=0.14187
seed=42 | final_epoch=04/07 | train_loss=0.13860
seed=42 | final_epoch=05/07 | train_loss=0.13665
seed=42 | final_epoch=06/07 | train_loss=0.13509
seed=42 | final_epoch=07/07 | train_loss=0.13401
  -> saved /kaggle/working/final_seed_42.pt
PHASE 2 | seed = 52 | final_epochs = 6
seed=52 | final_epoch=01/06 | train_loss=0.20193
seed=52 | final_epoch=02/06 | train_loss=0.14918
seed=52 | final_epoch=03/06 | train_loss=0.14207
seed=52 | final_epoch=04/06 | train_loss=0.13872
seed=52 | final_epoch=05/06 | train_loss=0.13661
seed=52 | final_epoch=06/06 | train_loss=0.13611
  -> saved /kaggle/working/final_seed_52.pt
PHASE 2 | seed = 62 | final_epochs = 7
seed=62 | final_epoch=01/07 | train_loss=0.20437
seed=62 | final_epoch=02/07 | train_loss=0.14871
seed=62 | final_epoch=05/07 | train_loss=0.13652
seed=62 | f

In [12]:
# =============================================================
# 12. Prepare test chunks (split on whitespace, track char IDs)
# =============================================================
ws_text    = ws_test_path.read_text(encoding="utf-8")
sample_sub = pd.read_csv(sample_sub_path)

test_chunks  = []
current_text = ""
current_ids  = []

for char_id, ch in enumerate(ws_text, start=1):
    if ch.isspace():
        if current_text:
            test_chunks.append((current_text, current_ids.copy()))
            current_text = ""; current_ids.clear()
        continue
    current_text += ch
    current_ids.append(char_id)

if current_text:
    test_chunks.append((current_text, current_ids.copy()))

print("total chunks        :", len(test_chunks))
print("sample submission   :", len(sample_sub))
print("first chunk preview :", test_chunks[0][0][:80])


total chunks        : 2067
sample submission   : 35182
first chunk preview : ที่ยังสถานการณ์ยังไม่คลี่คลายอาจส่งผลกระทบการค้าชายแดนไทย


In [13]:
# =============================================================
# 13. Ensemble — average logits from all final models, then decode
# =============================================================
test_ds     = WSTestDataset(test_chunks, stoi)
test_loader = DataLoader(test_ds, batch_size=CFG["valid_batch_size"], shuffle=False,
                         collate_fn=test_collate_fn, num_workers=CFG["num_workers"],
                         pin_memory=torch.cuda.is_available())

@torch.no_grad()
def ensemble_predict_logits(ckpt_paths, loader):
    logit_sums: Dict[int, np.ndarray] = {}
    meta:       Dict[int, tuple]      = {}

    for ckpt_path in ckpt_paths:
        print("loading:", ckpt_path)
        model = build_model()
        ckpt  = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state_dict"])
        model.eval()

        for x, t, lengths, mask, texts, original_ids, chunk_indices in loader:
            x = x.to(DEVICE, non_blocking=True)
            t = t.to(DEVICE, non_blocking=True)
            with make_autocast():
                logits = model(x, t, lengths)
            logits_cpu  = logits.detach().float().cpu().numpy()
            lengths_cpu = lengths.cpu().numpy()
            for i, seq_len in enumerate(lengths_cpu):
                chunk_idx = int(chunk_indices[i])
                arr = logits_cpu[i, :seq_len].copy()
                if chunk_idx not in logit_sums:
                    logit_sums[chunk_idx] = arr
                    meta[chunk_idx]       = (texts[i], original_ids[i])
                else:
                    logit_sums[chunk_idx] += arr

        del model, ckpt; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    rows       = []
    num_models = len(ckpt_paths)
    for chunk_idx in range(len(test_chunks)):
        avg_logits = logit_sums[chunk_idx] / num_models
        pred_ids   = constrained_decode_single(avg_logits)
        pred_tags  = [ID2LABEL[p] for p in pred_ids]
        _, ids     = meta[chunk_idx]
        assert len(pred_tags) == len(ids)
        rows.extend(zip(ids, pred_tags))
    return rows

pred_rows = ensemble_predict_logits(final_ckpts, test_loader)
print("total pred rows:", len(pred_rows))
pred_rows[:10]


loading: /kaggle/working/final_seed_42.pt
loading: /kaggle/working/final_seed_52.pt
loading: /kaggle/working/final_seed_62.pt
total pred rows: 35182


[(1, 'B_WORD'),
 (2, 'I_WORD'),
 (3, 'E_WORD'),
 (4, 'B_WORD'),
 (5, 'I_WORD'),
 (6, 'E_WORD'),
 (7, 'B_WORD'),
 (8, 'I_WORD'),
 (9, 'I_WORD'),
 (10, 'I_WORD')]

In [14]:
# =============================================================
# 14. Build & save submission.csv
# =============================================================
submission = pd.DataFrame(pred_rows, columns=["Id", "Predicted"])
submission = submission.sort_values("Id").reset_index(drop=True)

print("submission rows:", len(submission))
print("sample rows    :", len(sample_sub))
print("ID match       :", submission["Id"].equals(sample_sub["Id"]))

out_path = Path("/kaggle/working/submission.csv")
submission.to_csv(out_path, index=False)
print("Saved →", out_path)
submission.head(20)


submission rows: 35182
sample rows    : 35182
ID match       : True
Saved → /kaggle/working/submission.csv


,Id,Predicted
0,1,B_WORD
1,2,I_WORD
2,3,E_WORD
3,4,B_WORD
4,5,I_WORD
5,6,E_WORD
6,7,B_WORD
7,8,I_WORD
8,9,I_WORD
9,10,I_WORD
